# **Remerging Data Tables**

**Import packages and data**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

In [2]:
# Connect to sqlite file in directory
conn = sqlite3.connect("eicu_v2_0_1.sqlite3")

## **1. Create Target Feature Table First**

In [3]:
# Pull the features we need from each table to create the target variable
query = """
SELECT
    p.patientunitstayid,
    p.hospitaldischargestatus,
    p.unitdischargestatus,
    p.unitvisitnumber,
    apv.visitnumber,
    apv.readmit
FROM patient p
LEFT JOIN apachepredvar apv
    ON p.patientunitstayid = apv.patientunitstayid
"""
target_df = pd.read_sql(query, conn)

In [4]:
target_df.head()

,patientunitstayid,hospitaldischargestatus,unitdischargestatus,unitvisitnumber,visitnumber,readmit
0,141764,Alive,Alive,2,NaN,NaN
1,141765,Alive,Alive,1,1.0,0.0
2,143870,Alive,Alive,1,1.0,0.0
3,144815,Alive,Alive,1,1.0,0.0
4,145427,Alive,Alive,1,1.0,0.0


In [5]:
# Define bad_outcome composite results

# ICU death
target_df['ICU_death'] = (
    target_df['unitdischargestatus']
    .str.lower()
    .str.contains('expired', na=False)).astype(int)

# Hospital death
target_df['hospital_death'] = (
    target_df['hospitaldischargestatus']
    .str.lower()
    .str.contains('expired', na=False)).astype(int)

# ICU readmission 
target_df['ICU_readmit'] = (
    target_df['visitnumber'] > 1).astype(int)

target_df['ICU_readmit'] = (
    target_df['unitvisitnumber'] > 1).astype(int)

# Clean readmit flag from dataset
target_df['readmit_clean'] = (
    target_df['readmit'].fillna(0).astype(int))

# Final composite outcome - without hospital death
# target_df['bad_outcome'] = (
    # (target_df['ICU_death'] == 1) |
    # (target_df['ICU_readmit'] == 1) |
    # (target_df['readmit_clean'] == 1)).astype(int)

# Final composite outcome - with hospital death
target_df['bad_outcome'] = (
    (target_df['ICU_death'] == 1) |
    (target_df['hospital_death'] == 1) |
    (target_df['ICU_readmit'] == 1) |
    (target_df['readmit_clean'] == 1)).astype(int)

In [6]:
# See the counts 

print("ICU_death:")
print(target_df['ICU_death'].value_counts())

print("\nhospital_death:")
print(target_df['hospital_death'].value_counts())

print("\nICU_readmit:")
print(target_df['ICU_readmit'].value_counts())

print("\nreadmit_clean:")
print(target_df['readmit_clean'].value_counts())

ICU_death:
ICU_death
0    2394
1     126
Name: count, dtype: int64

hospital_death:
hospital_death
0    2308
1     212
Name: count, dtype: int64

ICU_readmit:
ICU_readmit
0    2119
1     401
Name: count, dtype: int64

readmit_clean:
readmit_clean
0    2422
1      98
Name: count, dtype: int64


In [7]:
print(target_df['bad_outcome'].value_counts())

bad_outcome
0    1938
1     582
Name: count, dtype: int64


## **2. Create Feature Tables Next**

For final merged DataFrame, merge variable dataframe's:
- allergy_agg
- base_aps_df
- base_apr_df
- base_pred_df
- base_diag_df
- hosp_df
- base_io_df
- base_past_df
- patient_df
- base_vital_df
- target_df (at the very end)

**Merging from allergy table**

In [8]:
allergy_df = pd.read_sql("""
SELECT patientunitstayid, allergyType
FROM allergy
""", conn)

In [9]:
unique_allergies = allergy_df["allergytype"].unique()
print(unique_allergies)

<StringArray>
['Non Drug', 'Drug']
Length: 2, dtype: str


In [10]:
# Create flags
allergy_df['drug_allergy'] = (allergy_df['allergytype'] == 'Drug').astype(int)
allergy_df['non_drug_allergy'] = (allergy_df['allergytype'] == 'Non Drug').astype(int)

In [11]:
# Aggregate per patient
allergy_agg = allergy_df.groupby('patientunitstayid').agg({
    'drug_allergy': 'max',
    'non_drug_allergy': 'max'
}).reset_index()
allergy_agg = allergy_agg.fillna(0)

In [12]:
allergy_agg.head()

,patientunitstayid,drug_allergy,non_drug_allergy
0,243097,1,1
1,244477,1,0
2,246997,1,0
3,249328,0,1
4,250073,1,1


**Merging from apacheApsVar table**

In [13]:
df = pd.read_sql("SELECT patientunitstayid FROM apacheApsVar", conn)
df['patientunitstayid'].nunique(), len(df)

# 2205 unique patients so some are missing, we can just add a missing aps flag

(2205, 2205)

In [14]:
aps_df = pd.read_sql("""
SELECT
    patientunitstayid,
    dialysis, wbc, respiratoryRate, sodium, heartrate, meanbp,
    ph, hematocrit, creatinine, albumin, pao2, pco2,
    bun, glucose, bilirubin, fio2
FROM apacheApsVar
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid
FROM patient
""", conn)

In [15]:
base_aps_df = base_df.merge(aps_df, on="patientunitstayid", how="left")
base_aps_df["aps_missing"] = base_aps_df[
    ["wbc", "sodium", "heartrate", "meanbp"]].isna().any(axis=1).astype(int)

In [16]:
base_aps_df.head()

,patientunitstayid,dialysis,wbc,respiratoryrate,sodium,heartrate,meanbp,ph,hematocrit,creatinine,albumin,pao2,pco2,bun,glucose,bilirubin,fio2,aps_missing
0,141764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,141765,0.0,10.2,39.0,139.0,88.0,108.0,-1.0,37.8,1.04,-1.0,-1.0,-1.0,28.0,61.0,-1.0,-1.0,0
2,143870,0.0,11.7,60.0,133.0,40.0,47.0,-1.0,34.1,1.14,-1.0,-1.0,-1.0,14.0,140.0,-1.0,-1.0,0
3,144815,0.0,7.9,6.0,141.0,131.0,61.0,-1.0,36.6,0.63,3.6,-1.0,-1.0,6.0,82.0,0.5,-1.0,0
4,145427,0.0,21.1,41.0,141.0,49.0,72.0,-1.0,40.4,1.05,-1.0,-1.0,-1.0,14.0,118.0,-1.0,-1.0,0


In [17]:
# Check
base_aps_df.shape
base_aps_df['patientunitstayid'].nunique()

2520

**Merging from apachePatientResult table**

In [18]:
apache_df = pd.read_sql("""
SELECT
    patientunitstayid,
    acutePhysiologyScore,
    apacheScore,
    predictedicumortality
FROM apachePatientResult
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid
FROM patient
""", conn)

In [19]:
apache_df['patientunitstayid'].nunique(), len(apache_df)

(1838, 3676)

In [20]:
apache_df = apache_df.groupby("patientunitstayid").agg({
    "acutephysiologyscore": "max",
    "apachescore": "max"
}).reset_index()

In [21]:
apache_df.shape

(1838, 3)

In [22]:
# Merge into full patient spine (IMPORTANT: left join)
base_apr_df = base_df.merge(apache_df, on="patientunitstayid", how="left")

In [23]:
# Create missingness indicator flag
base_apr_df["apache_missing"] = base_apr_df["apachescore"].isna().astype(int)

# Impute missing values (safe clinical default: median)
base_apr_df["acutephysiologyscore"] = base_apr_df["acutephysiologyscore"].fillna(
    base_apr_df["acutephysiologyscore"].median())

base_apr_df["apachescore"] = base_apr_df["apachescore"].fillna(
    base_apr_df["apachescore"].median())

In [24]:
# Final sanity checks
print(base_apr_df.shape)
print(base_apr_df["patientunitstayid"].nunique())
print(base_apr_df[["apache_missing"]].value_counts())

(2520, 4)
2520
apache_missing
0                 1838
1                  682
Name: count, dtype: int64


**Merging from apachePredVar table**

In [25]:
pred_df = pd.read_sql("""
SELECT
    patientunitstayid,
    aids,
    hepaticFailure,
    lymphoma,
    metastaticCancer,
    leukemia,
    immunosuppression,
    cirrhosis,
    diabetes,
    midur
FROM apachePredVar
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid
FROM patient
""", conn)

In [26]:
pred_df.head()

,patientunitstayid,aids,hepaticfailure,lymphoma,metastaticcancer,leukemia,immunosuppression,cirrhosis,diabetes,midur
0,141765,0,0,0,0,0,0,0,0,0
1,143870,0,0,0,0,0,0,0,0,0
2,144815,0,0,0,0,0,0,0,0,0
3,145427,0,0,0,0,0,0,0,0,0
4,147307,0,0,0,0,0,0,0,0,0


In [27]:
print(pred_df.shape)
print(pred_df["patientunitstayid"].nunique())

# Handle duplicates safely
pred_df = pred_df.groupby("patientunitstayid").max().reset_index()

(2205, 10)
2205


In [28]:
# Merge into full patient dataset (KEEP ALL PATIENTS)
base_pred_df = base_df.merge(pred_df, on="patientunitstayid", how="left")

In [29]:
binary_cols = [
    "aids", "hepaticfailure", "lymphoma",
    "metastaticcancer", "leukemia",
    "immunosuppression", "cirrhosis",
    "diabetes", "midur"]

# Missingness indicator (do this BEFORE filling if you want true missing signal)
base_pred_df["pred_missing"] = base_pred_df[binary_cols].isna().any(axis=1).astype(int)

# Fill missing values safely (binary assumption)
base_pred_df[binary_cols] = base_pred_df[binary_cols].fillna(0)

In [30]:
# Final sanity checks
print(base_pred_df.shape)

(2520, 11)


**Merging from diagnosis table**

In [31]:
diagnosis_df = pd.read_sql("""
SELECT
    patientunitstayid,
    diagnosisPriority
FROM diagnosis
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid
FROM patient
""", conn)

In [32]:
diagnosis_df.shape

(24978, 2)

In [33]:
# Clean categorical values
diagnosis_df["diagnosispriority"] = (
    diagnosis_df["diagnosispriority"]
    .fillna("unknown")
    .str.lower()
    .str.strip())

In [34]:
# Aggregate to patient level
diag_features = diagnosis_df.groupby("patientunitstayid").agg(
    total_diagnoses=("diagnosispriority", "count"),
    primary_count=("diagnosispriority", lambda x: (x == "primary").sum()),
    major_count=("diagnosispriority", lambda x: (x == "major").sum()),
    other_count=("diagnosispriority", lambda x: (x == "other").sum())
).reset_index()

In [35]:
# Add useful ratios (clinical burden signals)
diag_features["primary_ratio"] = diag_features["primary_count"] / diag_features["total_diagnoses"]
diag_features["major_ratio"] = diag_features["major_count"] / diag_features["total_diagnoses"]

In [36]:
# Merge into full dataset (DO NOT lose patients)
base_diag_df = base_df.merge(diag_features, on="patientunitstayid", how="left")

In [37]:
# Handle missing patients (no diagnosis records)
base_diag_df["total_diagnoses"] = base_diag_df["total_diagnoses"].fillna(0)
base_diag_df["primary_count"] = base_diag_df["primary_count"].fillna(0)
base_diag_df["major_count"] = base_diag_df["major_count"].fillna(0)
base_diag_df["other_count"] = base_diag_df["other_count"].fillna(0)

base_diag_df["primary_ratio"] = base_diag_df["primary_ratio"].fillna(0)
base_diag_df["major_ratio"] = base_diag_df["major_ratio"].fillna(0)

In [38]:
# Sanity check
base_diag_df.shape

(2520, 7)

**Merging from hospital table**

In [39]:
hospital_df = pd.read_sql("""
SELECT
    hospitalid,
    numbedscategory,
    teachingstatus,
    region
FROM hospital
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid, hospitalid
FROM patient
""", conn)

In [40]:
hosp_df = base_df.merge(hospital_df, on="hospitalid", how="left")

In [41]:
hosp_df["numbedscategory"] = hosp_df["numbedscategory"].fillna("unknown")
hosp_df["teachingstatus"] = hosp_df["teachingstatus"].fillna("unknown")
hosp_df["region"] = hosp_df["region"].fillna("unknown")

In [42]:
hosp_df.shape

(2520, 5)

**Merging from intakeoutput table**

In [43]:
io_df = pd.read_sql("""
SELECT
    patientunitstayid,
    intakeOutputOffset,
    intakeTotal,
    outputTotal,
    dialysisTotal,
    netTotal
FROM intakeoutput
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid
FROM patient
""", conn)

In [44]:
io_df.shape

(100466, 6)

In [45]:
# First 24 hours 
# 24 hours = 1440 minutes
io_df = io_df[io_df["intakeoutputoffset"] <= 1440]

In [46]:
io_features = io_df.groupby("patientunitstayid").agg({
    "intaketotal": "max",
    "outputtotal": "max",
    "dialysistotal": "max",
    "nettotal": "max"
}).reset_index()

In [47]:
base_io_df = base_df.merge(io_features, on="patientunitstayid", how="left")

In [48]:
# Missingness indicator
io_cols = ["intaketotal", "outputtotal", "dialysistotal", "nettotal"]

base_io_df["io_missing"] = base_io_df[io_cols].isna().any(axis=1).astype(int)

for col in io_cols:
    base_io_df[col] = base_io_df[col].fillna(base_io_df[col].median())

In [49]:
base_io_df.shape

(2520, 6)

**Merging from pasthistory table**

In [50]:
past_df = pd.read_sql("""
SELECT
    patientunitstayid,
    pasthistoryvalue
FROM pasthistory
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid
FROM patient
""", conn)

In [51]:
pasthistoryvals=past_df["pasthistoryvalue"].unique()
print(pasthistoryvals)

<StringArray>
[                  'No Health Problems',
                            'Performed',
                 'medication dependent',
                                 'lung',
                                'other',
         'chemotherapy within past mo.',
               'COPD  - no limitations',
                        'head and neck',
                         'primary site',
                        'CABG - remote',
 ...
 'respiratory failure - within 5 years',
                    'angina - class II',
                           'FEV1 61-70',
                 'FEV1/FVC ratio 41-50',
                  'sickle cell disease',
         'respiratory failure - remote',
                             'DLCO <30',
                            'FVC 31-40',
                             'FEV1 <30',
                          'scleroderma']
Length: 174, dtype: str


In [52]:
# Clean text
past_df["pasthistoryvalue"] = (
    past_df["pasthistoryvalue"]
    .fillna("")
    .str.lower()
    .str.strip())

In [53]:
# Map to clinical categories

def map_conditions(text):
    return pd.Series({
        "hx_cardio": int(any(k in text for k in ["chf", "mi", "angina", "cabg", "atrial", "ventricular", "paced"])),
        "hx_respiratory": int(any(k in text for k in ["copd", "asthma", "respiratory failure", "oxygen"])),
        "hx_neuro": int(any(k in text for k in ["stroke", "tia", "seizure", "dementia", "encephalopathy"])),
        "hx_cancer": int(any(k in text for k in ["cancer", "carcinoma", "leukemia", "lymphoma", "melanoma", "tumor"])),
        "hx_renal": int(any(k in text for k in ["renal", "kidney", "dialysis"])),
        "hx_liver": int(any(k in text for k in ["liver", "cirrho", "varices", "ascites"])),
        "hx_endocrine": int(any(k in text for k in ["diabetes", "thyroid"])),
        "hx_immuno": int(any(k in text for k in ["chemo", "steroid", "hiv", "transplant", "immuno"])),
        "hx_heme": int(any(k in text for k in ["anemia", "clot", "dvt", "pe", "sickle"])),
        "hx_none": int("no health problems" in text)})

# Apply mapping
mapped = past_df["pasthistoryvalue"].apply(map_conditions)
past_df = pd.concat([past_df, mapped], axis=1)

In [54]:
# Aggregate per patient
past_features = past_df.groupby("patientunitstayid").max().reset_index()

# Merge
base_past_df = base_df.merge(past_features, on="patientunitstayid", how="left")

# Fill missing
cols = [col for col in past_features.columns if col != "patientunitstayid"]
base_past_df[cols] = base_past_df[cols].fillna(0)

In [55]:
base_past_df.shape

(2520, 12)

**Merging from patient table**

In [56]:
patient_df = pd.read_sql("""
SELECT
    patientunitstayid,
    gender,
    age,
    ethnicity,
    admissionHeight,
    admissionWeight,
    unittype
FROM patient
""", conn)

In [57]:
patient_df.shape # Already good

(2520, 7)

In [58]:
# Turn number to numeric
patient_df["age"] = patient_df["age"].replace("> 89", 90)
patient_df["age"] = pd.to_numeric(patient_df["age"], errors="coerce")
patient_df["admissionheight"] = pd.to_numeric(patient_df["admissionheight"], errors="coerce")
patient_df["admissionweight"] = pd.to_numeric(patient_df["admissionweight"], errors="coerce")

# Add BMI
patient_df["bmi"] = patient_df["admissionweight"] / ((patient_df["admissionheight"] / 100) ** 2)

In [59]:
# FillNA - we can explore these options later

# patient_df["age"] = patient_df["age"].fillna(patient_df["age"].median())
# patient_df["admissionHeight"] = patient_df["admissionHeight"].fillna(patient_df["admissionHeight"].median())
# patient_df["admissionWeight"] = patient_df["admissionWeight"].fillna(patient_df["admissionWeight"].median())
# patient_df["bmi"] = patient_df["bmi"].fillna(patient_df["bmi"].median())

**Merging from vitalAperiodic table**

In [60]:
vital_df = pd.read_sql("""
SELECT
    patientunitstayid,
    observationOffset,
    temperature,
    heartrate,
    respiration,
    sao2,
    systemicSystolic,
    systemicDiastolic,
    systemicMean
FROM vitalPeriodic
""", conn)

base_df = pd.read_sql("""
SELECT patientunitstayid, hospitalid
FROM patient
""", conn)

In [61]:
vital_df = vital_df[vital_df["observationoffset"] <= 1440] # First 24 hours
vital_df.head()

,patientunitstayid,observationoffset,temperature,heartrate,respiration,sao2,systemicsystolic,systemicdiastolic,systemicmean
0,141765,1179,,82,,,,,
1,141765,189,,76,30,97,,,
2,141765,1169,,84,,,,,
4,141765,1164,,86,,,,,
5,141765,159,,80,22,96,,,


In [62]:
vital_df.dtypes

patientunitstayid     int64
observationoffset     int64
temperature          object
heartrate            object
respiration          object
sao2                 object
systemicsystolic     object
systemicdiastolic    object
systemicmean         object
dtype: object

In [63]:
vital_cols = [
    "temperature", "heartrate", "respiration", "sao2",
    "systemicsystolic", "systemicdiastolic", "systemicmean"]

for col in vital_cols:
    vital_df[col] = pd.to_numeric(vital_df[col], errors="coerce")

In [64]:
vital_features = vital_df.groupby("patientunitstayid").agg({
    "temperature": ["min", "max"],
    "heartrate": ["min", "max"],
    "respiration": "max",
    "sao2": "min",
    "systemicsystolic": "min",
    "systemicdiastolic": "min",
    "systemicmean": "min"
}).reset_index()

In [65]:
vital_features.columns = [
    "patientunitstayid",
    "temp_min", "temp_max",
    "hr_min", "hr_max",
    "resp_max",
    "sao2_min",
    "sbp_min",
    "dbp_min",
    "map_min"]

In [66]:
base_vital_df = base_df.merge(vital_features, on="patientunitstayid", how="left")

In [67]:
vital_cols = [
    "temp_min", "temp_max", "hr_min", "hr_max",
    "resp_max", "sao2_min", "sbp_min", "dbp_min", "map_min"]

base_vital_df["vitals_missing"] = base_vital_df[vital_cols].isna().any(axis=1).astype(int)
for col in vital_cols:
    base_vital_df[col] = base_vital_df[col].fillna(base_vital_df[col].median())

In [68]:
base_vital_df["hr_range"] = base_vital_df["hr_max"] - base_vital_df["hr_min"]
base_vital_df["temp_range"] = base_vital_df["temp_max"] - base_vital_df["temp_min"]

In [69]:
base_vital_df.shape

(2520, 14)

## **3. Create Final Merged DataFrame**

For final merged DataFrame, merge variable dataframe's:

- allergy_agg
- base_aps_df
- base_apr_df
- base_pred_df
- base_diag_df
- hosp_df
- base_io_df
- base_past_df
- patient_df
- base_vital_df
- target_df (at the very end)

In [70]:
# Check to see if all df's have patientunitstayid
hosp_df.head()

,patientunitstayid,hospitalid,numbedscategory,teachingstatus,region
0,141764,59,<100,f,Midwest
1,141765,59,<100,f,Midwest
2,143870,68,<100,f,Midwest
3,144815,56,<100,f,Midwest
4,145427,68,<100,f,Midwest


In [71]:
final_merged = patient_df.copy()

# Core clinical features
final_merged = final_merged.merge(base_aps_df, on="patientunitstayid", how="left")
final_merged = final_merged.merge(base_apr_df, on="patientunitstayid", how="left")
final_merged = final_merged.merge(base_pred_df, on="patientunitstayid", how="left")

# Diagnosis + history
final_merged = final_merged.merge(base_diag_df, on="patientunitstayid", how="left")
final_merged = final_merged.merge(base_past_df, on="patientunitstayid", how="left")

# Treatments / intake-output
final_merged = final_merged.merge(base_io_df, on="patientunitstayid", how="left")

# Allergy
final_merged = final_merged.merge(allergy_agg, on="patientunitstayid", how="left")

# Vitals (VERY important block)
final_merged = final_merged.merge(base_vital_df, on="patientunitstayid", how="left")

# Intake output
final_merged = final_merged.merge(base_io_df, on="patientunitstayid", how="left")

# Hospital (merge by hospitalid, not patientunitstayid)
final_merged = final_merged.merge(hosp_df, on="patientunitstayid", how="left")

# Target LAST
final_merged = final_merged.merge(target_df, on="patientunitstayid", how="left")

In [72]:
final_merged.head()

,patientunitstayid,gender,age,ethnicity,admissionheight,admissionweight,unittype,bmi,dialysis,wbc,...,hospitaldischargestatus,unitdischargestatus,unitvisitnumber,visitnumber,readmit,ICU_death,hospital_death,ICU_readmit,readmit_clean,bad_outcome
0,141764,Female,87.0,Caucasian,157.5,NaN,Med-Surg ICU,NaN,NaN,NaN,...,Alive,Alive,2,NaN,NaN,0,0,1,0,1
1,141765,Female,87.0,Caucasian,157.5,46.5,Med-Surg ICU,18.745276,0.0,10.2,...,Alive,Alive,1,1.0,0.0,0,0,0,0,0
2,143870,Male,76.0,Caucasian,167.0,77.5,SICU,27.788734,0.0,11.7,...,Alive,Alive,1,1.0,0.0,0,0,0,0,0
3,144815,Female,34.0,Caucasian,172.7,60.3,Med-Surg ICU,20.217741,0.0,7.9,...,Alive,Alive,1,1.0,0.0,0,0,0,0,0
4,145427,Male,61.0,Caucasian,177.8,91.7,SICU,29.007201,0.0,21.1,...,Alive,Alive,1,1.0,0.0,0,0,0,0,0


In [73]:
# Save it
final_merged.to_csv("final_merged.csv", index=False)